# CPU-Only HD Upscaler (quota-free)

**Set Accelerator = None** (Settings -> Accelerator -> None) so this uses **zero GPU hours**.

1. Add your `Beholder_Chunk_*.zip` as a **Dataset** (or upload into `/kaggle/input`).
2. **Run All**. HD clips + `hd_out.zip` appear on the **Output** tab.

- `SCALE = 2.0` -> 768x512 becomes **1536x1024** (same as take_0).
- `USE_AI = True` tries **EDSR** (sharper AI upscale); **auto-falls back to Lanczos** if the model/deps fail, so it always produces HD.
- Audio is preserved when a clip has it.


In [ ]:
# ============================================================================
# CPU-ONLY HD UPSCALER  --  set Accelerator = None (zero GPU-quota cost)
# 1) Add your Beholder_Chunk_*.zip as a Dataset (or upload into /kaggle/input)
# 2) Run All -> HD clips + hd_out.zip appear on the Output tab
# ============================================================================
import os, sys, glob, base64, subprocess, urllib.request

# ---------------- KNOBS ----------------
INPUT  = ""        # leave "" to auto-pick the newest .zip / .mp4s under /kaggle/input
SCALE  = 2.0       # 768x512 -> 1536x1024 (same as take_0)
USE_AI = True      # True = try EDSR (AI, sharper); auto-falls back to lanczos if it fails
OUT    = "/kaggle/working/hd_out"

# ---------------- 1) write the tested upscale engine ----------------
_ENGINE_B64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJ1cHNjYWxlX2NsaXBzLnB5IC0tIGJhdGNoIHVwc2NhbGUgcmVuZGVyZWQgY2xpcHMgdG8gSEQgKENQVSwgcXVvdGEtZnJlZSkuCgpSdW5zIGFueXdoZXJlIHdpdGggZmZtcGVnOiBsb2NhbGx5IG9uIHlvdXIgUEMsIE9SIGluIGEgQ1BVLW9ubHkgS2FnZ2xlIG5vdGVib29rCihrZWVwIHRoZSBhY2NlbGVyYXRvciBzZXQgdG8gKk5vbmUqIHNvIGl0IGRvZXMgTk9UIGJ1cm4geW91ciAzMCBHUFUtaHIgcXVvdGEpLgoKVXNhZ2U6CiAgICBweXRob24gdXBzY2FsZV9jbGlwcy5weSBJTiBPVVQgW1NDQUxFXQogICAgICBJTiAgICA6IGEgZm9sZGVyIG9mIC5tcDRzLCBPUiBhIC56aXAgb2YgY2xpcHMsIE9SIGEgc2luZ2xlIC5tcDQKICAgICAgT1VUICAgOiBvdXRwdXQgZm9sZGVyIChjcmVhdGVkKTsgYW4gPE9VVD4uemlwIGlzIGFsc28gd3JpdHRlbgogICAgICBTQ0FMRSA6IHVwc2NhbGUgZmFjdG9yIChkZWZhdWx0IDIuMCAtPiA3Njh4NTEyIGJlY29tZXMgMTUzNngxMDI0KQoKUXVhbGl0eSB0aWVyIChlbnYgVVBTQ0FMRV9NRVRIT0QpOgogICAgbGFuY3pvcyAgICAoZGVmYXVsdCkgOiBmZm1wZWcgTGFuY3pvcy4gUHJvdmVuLCBmYXN0LCBwcmVzZXJ2ZXMgYXVkaW8uIFNvZnRlciB0aGFuIEFJLgogICAgZWRzciAgICAgICAgICAgICAgICAgOiBPcGVuQ1YgZG5uX3N1cGVycmVzIEVEU1IgKEFJLCBDUFUpLiBTaGFycGVyLiBOZWVkcyBFRFNSX3h7MiwzLDR9LnBiCiAgICAgICAgICAgICAgICAgICAgICAgICAgIChzZXQgRURTUl9NT0RFTD0vcGF0aC90by9FRFNSX3gyLnBiKS4KQW55IEFJIHRpZXIgdGhhdCBmYWlscyBvbiBhIGNsaXAgQVVUT01BVElDQUxMWSBmYWxscyBiYWNrIHRvIGxhbmN6b3MsIHNvIHRoZSB0b29sCmFsd2F5cyBwcm9kdWNlcyBhbiBIRCBmaWxlLiBBdWRpbyBpcyBwcmVzZXJ2ZWQgd2hlbiBwcmVzZW50LgoiIiIKaW1wb3J0IG9zLCBzeXMsIGdsb2IsIHNodXRpbCwgc3VicHJvY2VzcywgdGVtcGZpbGUsIHppcGZpbGUKCmRlZiBfcnVuKGNtZCwgY2hlY2s9VHJ1ZSk6CiAgICByZXR1cm4gc3VicHJvY2Vzcy5ydW4oY21kLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsIGNoZWNrPWNoZWNrKQoKZGVmIGhhc19hdWRpbyhwYXRoKToKICAgIHIgPSBfcnVuKFsiZmZwcm9iZSIsIi12IiwiZXJyb3IiLCItc2VsZWN0X3N0cmVhbXMiLCJhIiwKICAgICAgICAgICAgICAiLXNob3dfZW50cmllcyIsInN0cmVhbT1pbmRleCIsIi1vZiIsImNzdj1wPTAiLHBhdGhdLCBjaGVjaz1GYWxzZSkKICAgIHJldHVybiBib29sKHIuc3Rkb3V0LnN0cmlwKCkpCgpkZWYgcHJvYmVfd2gocGF0aCk6CiAgICByID0gX3J1bihbImZmcHJvYmUiLCItdiIsImVycm9yIiwiLXNlbGVjdF9zdHJlYW1zIiwidjowIiwKICAgICAgICAgICAgICAiLXNob3dfZW50cmllcyIsInN0cmVhbT13aWR0aCxoZWlnaHQiLCItb2YiLCJjc3Y9cD0wOnM9eCIscGF0aF0sIGNoZWNrPUZhbHNlKQogICAgdHJ5OgogICAgICAgIHcsaCA9IHIuc3Rkb3V0LnN0cmlwKCkuc3BsaXQoIngiKTsgcmV0dXJuIGludCh3KSwgaW50KGgpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBOb25lLCBOb25lCgpkZWYgX2ZwcyhwYXRoKToKICAgIHIgPSBfcnVuKFsiZmZwcm9iZSIsIi12IiwiZXJyb3IiLCItc2VsZWN0X3N0cmVhbXMiLCJ2OjAiLCItc2hvd19lbnRyaWVzIiwKICAgICAgICAgICAgICAic3RyZWFtPXJfZnJhbWVfcmF0ZSIsIi1vZiIsImRlZmF1bHQ9bnc9MTpuaz0xIixwYXRoXSwgY2hlY2s9RmFsc2UpCiAgICB0cnk6CiAgICAgICAgYSxiID0gci5zdGRvdXQuc3RyaXAoKS5zcGxpdCgiLyIpOyByZXR1cm4gZmxvYXQoYSkvZmxvYXQoYikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIDI0LjAKCmRlZiB1cHNjYWxlX2xhbmN6b3MoaW5wLCBvdXRwLCBzY2FsZSk6CiAgICB2ZiA9ICJzY2FsZT0ndHJ1bmMoaXcqJXMvMikqMic6J3RydW5jKGloKiVzLzIpKjInOmZsYWdzPWxhbmN6b3MiICUgKHNjYWxlLCBzY2FsZSkKICAgIGNtZCA9IFsiZmZtcGVnIiwiLXkiLCItbG9nbGV2ZWwiLCJlcnJvciIsIi1pIixpbnAsIi12ZiIsdmYsCiAgICAgICAgICAgIi1jOnYiLCJsaWJ4MjY0IiwiLXByZXNldCIsInNsb3ciLCItY3JmIiwiMTIiLCItcGl4X2ZtdCIsInl1djQyMHAiXQogICAgY21kICs9IChbIi1jOmEiLCJjb3B5Il0gaWYgaGFzX2F1ZGlvKGlucCkgZWxzZSBbIi1hbiJdKQogICAgY21kICs9IFtvdXRwXQogICAgX3J1bihjbWQpCgpkZWYgdXBzY2FsZV9lZHNyKGlucCwgb3V0cCwgc2NhbGUpOgogICAgaW1wb3J0IGN2MgogICAgcyA9IGludChyb3VuZChzY2FsZSkpCiAgICBpZiBzIG5vdCBpbiAoMiwzLDQpOiByYWlzZSBWYWx1ZUVycm9yKCJFRFNSIHN1cHBvcnRzIHgyL3gzL3g0IG9ubHkiKQogICAgbW9kZWwgPSBvcy5lbnZpcm9uLmdldCgiRURTUl9NT0RFTCIsCiAgICAgICAgICAgIG9zLnBhdGguam9pbihvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSksICJFRFNSX3glZC5wYiIgJSBzKSkKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhtb2RlbCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoIkVEU1IgbW9kZWwgbWlzc2luZzogJXMiICUgbW9kZWwpCiAgICBzciA9IGN2Mi5kbm5fc3VwZXJyZXMuRG5uU3VwZXJSZXNJbXBsX2NyZWF0ZSgpCiAgICBzci5yZWFkTW9kZWwobW9kZWwpOyBzci5zZXRNb2RlbCgiZWRzciIsIHMpCiAgICB0bXAgPSB0ZW1wZmlsZS5ta2R0ZW1wKCkKICAgIHRyeToKICAgICAgICBmcHMgPSBfZnBzKGlucCkKICAgICAgICBfcnVuKFsiZmZtcGVnIiwiLXkiLCItbG9nbGV2ZWwiLCJlcnJvciIsIi1pIixpbnAsIG9zLnBhdGguam9pbih0bXAsImZfJTA2ZC5wbmciKV0pCiAgICAgICAgZnJhbWVzID0gc29ydGVkKGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4odG1wLCJmXyoucG5nIikpKQogICAgICAgIGlmIG5vdCBmcmFtZXM6IHJhaXNlIFJ1bnRpbWVFcnJvcigibm8gZnJhbWVzIGV4dHJhY3RlZCIpCiAgICAgICAgZm9yIGYgaW4gZnJhbWVzOgogICAgICAgICAgICBjdjIuaW13cml0ZShmLCBzci51cHNhbXBsZShjdjIuaW1yZWFkKGYsIGN2Mi5JTVJFQURfQ09MT1IpKSwKICAgICAgICAgICAgICAgICAgICAgICAgW2N2Mi5JTVdSSVRFX1BOR19DT01QUkVTU0lPTiwxXSkKICAgICAgICB2dG1wID0gb3MucGF0aC5qb2luKHRtcCwidi5tcDQiKQogICAgICAgIF9ydW4oWyJmZm1wZWciLCIteSIsIi1sb2dsZXZlbCIsImVycm9yIiwiLWZyYW1lcmF0ZSIsIiUuNGYiJWZwcywKICAgICAgICAgICAgICAiLWkiLG9zLnBhdGguam9pbih0bXAsImZfJTA2ZC5wbmciKSwiLWM6diIsImxpYngyNjQiLCItcHJlc2V0Iiwic2xvdyIsCiAgICAgICAgICAgICAgIi1jcmYiLCIxMiIsIi1waXhfZm10IiwieXV2NDIwcCIsIHZ0bXBdKQogICAgICAgIGlmIGhhc19hdWRpbyhpbnApOgogICAgICAgICAgICBfcnVuKFsiZmZtcGVnIiwiLXkiLCItbG9nbGV2ZWwiLCJlcnJvciIsIi1pIix2dG1wLCItaSIsaW5wLAogICAgICAgICAgICAgICAgICAiLW1hcCIsIjA6djowIiwiLW1hcCIsIjE6YTowIiwiLWM6diIsImNvcHkiLCItYzphIiwiY29weSIsIi1zaG9ydGVzdCIsb3V0cF0pCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2h1dGlsLm1vdmUodnRtcCwgb3V0cCkKICAgIGZpbmFsbHk6CiAgICAgICAgc2h1dGlsLnJtdHJlZSh0bXAsIGlnbm9yZV9lcnJvcnM9VHJ1ZSkKCk1FVEhPRFMgPSB7ImxhbmN6b3MiOiB1cHNjYWxlX2xhbmN6b3MsICJlZHNyIjogdXBzY2FsZV9lZHNyfQoKZGVmIHByb2Nlc3Nfb25lKGlucCwgb3V0cCwgc2NhbGUsIG1ldGhvZCk6CiAgICBmbiA9IE1FVEhPRFMuZ2V0KG1ldGhvZCwgdXBzY2FsZV9sYW5jem9zKQogICAgdHJ5OgogICAgICAgIGZuKGlucCwgb3V0cCwgc2NhbGUpOyByZXR1cm4gbWV0aG9kCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgaWYgbWV0aG9kICE9ICJsYW5jem9zIjoKICAgICAgICAgICAgcHJpbnQoIiAgIFslcyBmYWlsZWQ6ICVyXSAtPiBmYWxsaW5nIGJhY2sgdG8gbGFuY3pvcyIgJSAobWV0aG9kLCBlKSkKICAgICAgICAgICAgdXBzY2FsZV9sYW5jem9zKGlucCwgb3V0cCwgc2NhbGUpOyByZXR1cm4gImxhbmN6b3MoZmFsbGJhY2spIgogICAgICAgIHJhaXNlCgpkZWYgZ2F0aGVyX2lucHV0cyhzcmMsIHdvcmtkaXIpOgogICAgaWYgb3MucGF0aC5pc2RpcihzcmMpOgogICAgICAgIHJldHVybiBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbihzcmMsIioqIiwiKi5tcDQiKSwgcmVjdXJzaXZlPVRydWUpKQogICAgaWYgc3JjLmxvd2VyKCkuZW5kc3dpdGgoIi56aXAiKToKICAgICAgICBleCA9IG9zLnBhdGguam9pbih3b3JrZGlyLCJfdW56aXBwZWQiKTsgb3MubWFrZWRpcnMoZXgsIGV4aXN0X29rPVRydWUpCiAgICAgICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoc3JjKSBhcyB6OiB6LmV4dHJhY3RhbGwoZXgpCiAgICAgICAgcmV0dXJuIHNvcnRlZChnbG9iLmdsb2Iob3MucGF0aC5qb2luKGV4LCIqKiIsIioubXA0IiksIHJlY3Vyc2l2ZT1UcnVlKSkKICAgIGlmIHNyYy5sb3dlcigpLmVuZHN3aXRoKCIubXA0Iik6CiAgICAgICAgcmV0dXJuIFtzcmNdCiAgICByZXR1cm4gW10KCmRlZiBtYWluKCk6CiAgICBpZiBsZW4oc3lzLmFyZ3YpIDwgMzoKICAgICAgICBwcmludChfX2RvY19fKTsgc3lzLmV4aXQoMSkKICAgIHNyYywgb3V0ID0gc3lzLmFyZ3ZbMV0sIHN5cy5hcmd2WzJdCiAgICBzY2FsZSA9IGZsb2F0KHN5cy5hcmd2WzNdKSBpZiBsZW4oc3lzLmFyZ3YpID4gMyBlbHNlIDIuMAogICAgbWV0aG9kID0gb3MuZW52aXJvbi5nZXQoIlVQU0NBTEVfTUVUSE9EIiwibGFuY3pvcyIpLmxvd2VyKCkKICAgIG9zLm1ha2VkaXJzKG91dCwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGNsaXBzID0gZ2F0aGVyX2lucHV0cyhzcmMsIG91dCkKICAgIGlmIG5vdCBjbGlwczoKICAgICAgICBwcmludCgiTm8gLm1wNCBmaWxlcyBmb3VuZCBpbiIsIHNyYyk7IHN5cy5leGl0KDEpCiAgICBwcmludCgiVXBzY2FsaW5nICVkIGNsaXAocykgeCUuMmYgdmlhICclcycgKGF1ZGlvIHByZXNlcnZlZCkgLi4uIiAlIChsZW4oY2xpcHMpLCBzY2FsZSwgbWV0aG9kKSkKICAgIGZvciBpLCBjIGluIGVudW1lcmF0ZShjbGlwcywgMSk6CiAgICAgICAgdyxoID0gcHJvYmVfd2goYykKICAgICAgICBkc3QgPSBvcy5wYXRoLmpvaW4ob3V0LCBvcy5wYXRoLnNwbGl0ZXh0KG9zLnBhdGguYmFzZW5hbWUoYykpWzBdICsgIl9oZC5tcDQiKQogICAgICAgIHVzZWQgPSBwcm9jZXNzX29uZShjLCBkc3QsIHNjYWxlLCBtZXRob2QpCiAgICAgICAgbncsbmggPSBwcm9iZV93aChkc3QpCiAgICAgICAgcHJpbnQoIiAgWyVkLyVkXSAlLTI4cyAlc3glcyAtPiAlc3glcyAgWyVzXSIgJSAoaSwgbGVuKGNsaXBzKSwgb3MucGF0aC5iYXNlbmFtZShjKSwgdyxoLG53LG5oLCB1c2VkKSkKICAgIHpiYXNlID0gb3V0LnJzdHJpcCgiL1xcIikKICAgIHNodXRpbC5tYWtlX2FyY2hpdmUoemJhc2UsICJ6aXAiLCBvdXQpCiAgICBwcmludCgiRG9uZSAtPiAlcyAgYW5kICAlcy56aXAiICUgKG91dCwgemJhc2UpKQoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="
with open("/kaggle/working/upscale_clips.py", "wb") as _f:
    _f.write(base64.b64decode(_ENGINE_B64))
print("[setup] wrote upscale_clips.py")

# ---------------- 2) deps (ffmpeg preinstalled; ensure cv2 dnn_superres) ----------------
def _sh(c): return subprocess.run(c, capture_output=True, text=True)
if USE_AI:
    try:
        import cv2
        cv2.dnn_superres.DnnSuperResImpl_create()
    except Exception:
        print("[setup] installing opencv-contrib-python for the AI tier ...")
        _sh([sys.executable, "-m", "pip", "install", "-q", "opencv-contrib-python"])

# ---------------- 3) fetch EDSR x2 model (fallback-safe) ----------------
MODEL = "/kaggle/working/EDSR_x2.pb"
if USE_AI and not os.path.exists(MODEL):
    for _u in [
        "https://github.com/Saafke/EDSR_Tensorflow/raw/master/models/EDSR_x2.pb",
        "https://raw.githubusercontent.com/Saafke/EDSR_Tensorflow/master/models/EDSR_x2.pb",
    ]:
        try:
            print("[setup] downloading EDSR model:", _u)
            urllib.request.urlretrieve(_u, MODEL)
            if os.path.getsize(MODEL) > 100000:
                print("[setup] EDSR model ready (%d bytes)" % os.path.getsize(MODEL)); break
        except Exception as _e:
            print("[setup] model download failed:", repr(_e))
    else:
        print("[setup] no EDSR model -> will use lanczos (still full HD)")

# ---------------- 4) pick input ----------------
if not INPUT:
    _c = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True), key=os.path.getmtime, reverse=True)
    if not _c and glob.glob("/kaggle/input/**/*.mp4", recursive=True):
        _c = ["/kaggle/input"]
    INPUT = _c[0] if _c else ""
print("[run] INPUT =", INPUT or "(none found)")
assert INPUT, "No .zip or .mp4 found under /kaggle/input -- add your chunk zip as a Dataset."

# ---------------- 5) run ----------------
_env = dict(os.environ)
_use_edsr = USE_AI and os.path.exists(MODEL)
_env["UPSCALE_METHOD"] = "edsr" if _use_edsr else "lanczos"
_env["EDSR_MODEL"] = MODEL
print("[run] method =", _env["UPSCALE_METHOD"], "| scale =", SCALE)
_r = subprocess.run([sys.executable, "/kaggle/working/upscale_clips.py", INPUT, OUT, str(SCALE)], env=_env)
print("[done] rc =", _r.returncode, "-> HD zip at /kaggle/working/hd_out.zip (Output tab)")
